# Prediksi Keberhasilan Pemasaran Deposito Berjangka | K-Nearest Neighbors (KNN)
---


## 1. Business Understanding

### 1.1 Latar Belakang, Masalah, dan Solusi
1. **Latar Belakang**: Sebuah perusahaan perbankan besar memiliki kegiatan kampanye pemasaran telemarketing (*direct marketing campaigns*) untuk menawarkan produk deposito berjangka panjang kepada nasabah dengan menghubunginya secara langsung via telepon. Institusi memiliki data historis profil demografi dan riwayat interaksi nasabah.
2. **Masalah**: Proses telemarketing selama ini masih dilakukan secara manual dengan menghubungi nasabah secara acak (*random calling*). Pendekatan ini menghasilkan tingkat konversi yang rendah (hanya $\approx 11.7\%$ nasabah yang setuju berlangganan) serta membuang banyak waktu tenaga kerja dan biaya pulsa telepon operasional perbankan.
3. **Solusi & Tujuan**: Membangun model *machine learning* berbasis algoritma **K-Nearest Neighbors (KNN)** dengan memanfaatkan data historis profil demografi dan riwayat interaksi nasabah guna mengklasifikasikan apakah seorang nasabah akan berlangganan deposito (`yes` = 1) atau tidak (`no` = 0). Dengan demikian, tim telemarketing dapat memprioritaskan panggilan hanya kepada nasabah yang memiliki potensi konversi tinggi secara tepat sasaran, efisien, dan objektif.

### 1.2 Deskripsi Kolom
| Nama Kolom | Tipe Data | Keterangan & Definisi Variabel |
| :--- | :---: | :--- |
| **`Id`** | `int64` | Identitas unik atau nomor ID setiap baris data nasabah. |
| **`age`** | `numeric` | Umur nasabah dalam satuan tahun. |
| **`job`** | `categorical` | Jenis pekerjaan nasabah (*management*, *technician*, *blue-collar*, dll.). |
| **`marital`** | `categorical` | Status perkawinan nasabah (*married*, *single*, *divorced*). |
| **`education`** | `categorical` | Jenjang pendidikan terakhir nasabah (*primary*, *secondary*, *tertiary*, *unknown*). |
| **`default`** | `categorical` | Riwayat kredit macet (*yes*, *no*). |
| **`balance`** | `numeric` | Rata-rata saldo tahunan nasabah dalam rekening (dalam satuan Euro). |
| **`housing`** | `categorical` | Apakah nasabah memiliki cicilan pinjaman rumah / KPR (*yes*, *no*). |
| **`loan`** | `categorical` | Apakah nasabah memiliki pinjaman pribadi (*yes*, *no*). |
| **`contact`** | `categorical` | Jenis media komunikasi kontak (*cellular*, *telephone*, *unknown*). |
| **`day`** | `numeric` | Hari dalam bulan saat nasabah terakhir kali dihubungi (tanggal 1 s.d. 31). |
| **`month`** | `categorical` | Bulan dalam tahun saat nasabah terakhir kali dihubungi (*jan*, *feb*, dst.). |
| **`duration`** | `numeric` | Durasi percakapan panggilan telepon terakhir dalam satuan detik. |
| **`campaign`** | `numeric` | Jumlah kontak atau panggilan yang dilakukan selama kampanye ini. |
| **`pdays`** | `numeric` | Jumlah hari berlalu setelah dihubungi pada kampanye sebelumnya (-1 = belum pernah). |
| **`previous`** | `numeric` | Jumlah kontak interaksi yang dilakukan sebelum kampanye saat ini. |
| **`poutcome`** | `categorical` | Hasil status dari kampanye pemasaran sebelumnya (*success*, *failure*, *other*, *unknown*). |
| **`y`** | `categorical` | Keputusan nasabah berlangganan deposito berjangka (*yes* atau *no*). |

---

## 2. Data Preparation

Tahap *Data Preparation* difokuskan untuk memuat berkas dataset ke dalam memori kerja Python menggunakan pustaka `pandas`, memeriksa struktur data, membersihkan spasi  kolom, mengonversi fitur numerik yang belum terformat, serta memvalidasi dimensi, jumlah missing value, dan keberadaan duplikasi data.

### 2.1 Import Library

In [ ]:
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

### 2.2 Memuat Dataset
Memuat dataset `Assignment-2_Data.csv` dengan argumen `low_memory=False` dan membersihkan spasi di awal/akhir nama kolom.

In [ ]:
df = pd.read_csv('Assignment-2_Data.csv')
df.columns = df.columns.str.strip()

### 2.3 Menampilkan Data Teratas dan Terbawah

In [ ]:
df.head()

In [ ]:
df.tail()

### 2.4 Menampilkan Informasi Data

In [ ]:
df.info()

### 2.5 Merubah Kolom `age` dan `balance` Menjadi Numerik
Karena kolom **`age`** dan **`balance`** merupakan fitur numerik yang pada data mentah tersimpan sebagai tipe objek/string, kedua kolom harus dikonversi ke tipe data numerik (`Int64`). Nilai teks yang tidak valid diubah menjadi `NaN` menggunakan parameter `errors='coerce'`.

In [ ]:
df['age'] = pd.to_numeric(df['age'], errors='coerce')
df['age'] = df['age'].astype('Int64')
df['balance'] = pd.to_numeric(df['balance'], errors='coerce')
df['balance'] = df['balance'].astype('Int64')

In [ ]:
df.info()

### 2.6 Menampilkan dan Memeriksa Dimensi Data, Jumlah Data, Missing Value, dan Duplikasi
Memeriksa dimensi data (baris, kolom), total elemen data, sebaran nilai hilang (*missing value*), dan baris duplikat eksak.

In [ ]:
print(f'Dimensi Data: {df.shape}')
print(f'Jumlah Data: {df.size}')

print(f'\njumlah missing value: {df.isnull().sum().sum()}')
print(f'\n==== Jumlah Missing Value Per Kolom ====')
print(df.isnull().sum())

print(f'\nJumlah data duplikat: {df.duplicated().sum()}')

---

## 3. Exploratory Data Analysis (EDA)

Tahap *Exploratory Data Analysis (EDA)* dilakukan untuk memahami karakteristik statistik fitur numerik, mendeteksi keberadaan pencilan (*outlier*) metode IQR dan Z-Score, menganalisis korelasi antar-variabel, serta memvisualisasikan hubungan antara profil nasabah dan keputusan berlangganan deposito (`y`).

### 3.1 Menampilkan Deskripsi Data

In [ ]:
df.describe()

### 3.2 Analisis Outlier (Metode IQR dan Z-Score) pada Kolom Numerik

In [ ]:
outlier = []
numeric_columns = df.select_dtypes(include=[np.number]).columns.tolist()

for col in numeric_columns:
    if col in df.columns:
        q1 = df[col].quantile(0.25)
        q3 = df[col].quantile(0.75)
        median = df[col].median()
        iqr = q3 - q1
        lower = q1 - 1.5 * iqr
        upper = q3 + 1.5 * iqr
        iqr_outlier = df[(df[col] < lower) | (df[col] > upper)]
        mean = df[col].mean()
        std = df[col].std()
        z_score = (df[col] - mean) / std
        z_score_outlier = df[np.abs(z_score) > 3]
        
        outlier.append({
            'Kolom': col,
            'Mean': mean,
            'Standarisasi': std,
            'Min': df[col].min(),
            'Q1': q1,
            'Median': median,
            'Q3': q3,
            'Max': df[col].max(),
            'Batas Bawah IQR': lower,
            'Batas Atas IQR': upper,
            'Jumlah Outlier IQR': iqr_outlier.shape[0],
            'Jumlah Outlier Z-Score': z_score_outlier.shape[0]
        })

pd.DataFrame(outlier)

### 3.3 Visualisasi Boxplot untuk Deteksi Outlier

In [ ]:
plt.figure(figsize=(12, 6), dpi=110)
sns.boxplot(data=df[['age', 'balance', 'duration', 'campaign']], palette=['lightblue', 'moccasin', 'lightgreen', 'lightcoral'])
plt.title('Boxplot Umur, Saldo, Durasi, dan Kampanye', fontsize=13, fontweight='bold', pad=12)
plt.ylabel('Nilai', fontsize=11)
plt.tight_layout()
plt.show()

### 3.4 Visualisasi Distribusi Data Fitur Numerik

In [ ]:
fig, axes = plt.subplots(1, 4, figsize=(24, 5.5), dpi=110)

sns.histplot(df['age'].dropna(), bins=20, kde=True, ax=axes[0], color='tab:blue', edgecolor='white')
axes[0].set_title('Distribusi Umur', fontsize=13, fontweight='bold', pad=10)
axes[0].set_xlabel('Umur (Tahun)', fontsize=11)
axes[0].set_ylabel('Frekuensi', fontsize=11)

sns.histplot(df['balance'].dropna(), bins=20, ax=axes[1], color='tab:orange', edgecolor='white')
axes[1].set_title('Distribusi Saldo', fontsize=13, fontweight='bold', pad=10)
axes[1].set_xlabel('Saldo Rekening', fontsize=11)
axes[1].set_ylabel('Frekuensi', fontsize=11)

sns.histplot(df['duration'].dropna(), bins=20, ax=axes[2], color='tab:green', edgecolor='white')
axes[2].set_title('Distribusi Durasi', fontsize=13, fontweight='bold', pad=10)
axes[2].set_xlabel('Durasi Kontak (Detik)', fontsize=11)
axes[2].set_ylabel('Frekuensi', fontsize=11)

sns.histplot(df['campaign'].dropna(), bins=20, ax=axes[3], color='tab:red', edgecolor='white')
axes[3].set_title('Distribusi Kampanye', fontsize=13, fontweight='bold', pad=10)
axes[3].set_xlabel('Jumlah Kontak Kampanye', fontsize=11)
axes[3].set_ylabel('Frekuensi', fontsize=11)

plt.tight_layout()
plt.show()

### 3.5 Analisis Metode Korelasi Pearson, Spearman, dan Kendall

In [ ]:
var_list = ['age', 'balance', 'duration']

for col in var_list:
    df[col] = pd.to_numeric(df[col], errors='coerce')

korelasi_list = []
for i in range(len(var_list)):
    for j in range(i + 1, len(var_list)):
        var1 = var_list[i]
        var2 = var_list[j]
        clean_df = df[[var1, var2]].dropna()
        
        pearson_corr = clean_df[var1].corr(clean_df[var2], method='pearson')
        spearman_corr = clean_df[var1].corr(clean_df[var2], method='spearman')
        kendall_corr = clean_df[var1].corr(clean_df[var2], method='kendall')
        
        korelasi_list.append({
            'pasangan variabel': f'{var1} vs {var2}',
            'pearson correlation': pearson_corr,
            'spearman correlation': spearman_corr,
            'kendall correlation': kendall_corr
        })

pd.DataFrame(korelasi_list)

### 3.6 Visualisasi Heatmap Korelasi

In [ ]:
numeric_df = df.select_dtypes(include=[np.number]).drop(columns=['Id'], errors='ignore')
corr_matrix = numeric_df.corr(method='pearson')

plt.figure(figsize=(10, 7), dpi=110)
sns.heatmap(corr_matrix, annot=True, cmap='coolwarm', fmt='.2f', linewidths=0.5, cbar_kws={'label': 'Koefisien Korelasi'})
plt.title('Matriks Korelasi Pearson (Variabel Numerik)', fontsize=13, fontweight='bold', pad=12)
plt.tight_layout()
plt.show()

In [ ]:
df_encoded = df.copy()
for col in df_encoded.select_dtypes(include=['object', 'string']).columns:
    df_encoded[col] = df_encoded[col].astype('category').cat.codes

df_encoded = df_encoded.drop(columns=['Id'], errors='ignore')
corr_all = df_encoded.corr(method='pearson')

plt.figure(figsize=(14, 10), dpi=110)
sns.heatmap(corr_all, annot=True, cmap='coolwarm', fmt='.2f', linewidths=0.5, cbar_kws={'label': 'Koefisien Korelasi'})
plt.title('Matriks Korelasi Pearson (Seluruh Variabel)', fontsize=13, fontweight='bold', pad=12)
plt.tight_layout()
plt.show()

### 3.7 Visualisasi Scatter Plot Antar-Fitur Berdasarkan Keputusan Nasabah (`y`)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(20, 6), dpi=110)

sns.scatterplot(data=df, x='age', y='balance', hue='y', ax=axes[0], alpha=0.6, palette=['tab:blue', 'tab:red'])
axes[0].set_title('Umur vs Saldo (Berdasarkan Keputusan Deposito)', fontsize=13, fontweight='bold', pad=10)
axes[0].set_xlabel('Umur (Tahun)', fontsize=11)
axes[0].set_ylabel('Saldo Rekening (Balance)', fontsize=11)
axes[0].legend(title='Berlangganan (y)', frameon=True, facecolor='white')

sns.scatterplot(data=df, x='age', y='duration', hue='y', ax=axes[1], alpha=0.6, palette=['tab:blue', 'tab:red'])
axes[1].set_title('Umur vs Durasi Kontak (Berdasarkan Keputusan Deposito)', fontsize=13, fontweight='bold', pad=10)
axes[1].set_xlabel('Umur (Tahun)', fontsize=11)
axes[1].set_ylabel('Durasi Kontak Telepon (Detik)', fontsize=11)
axes[1].legend(title='Berlangganan (y)', frameon=True, facecolor='white')

plt.tight_layout()
plt.show()

---

## 4. Data Pre-Processing

Tahap *Data Pre-Processing* difokuskan untuk menyiapkan data agar optimal bagi algoritma K-Nearest Neighbors (KNN):
1. Menghapus fitur identitas unik (`Id`) yang tidak memiliki bobot prediktif.
2. Menghapus baris kosong awal (*missing value*).
3. Mengatasi anomali usia (`age` di luar rentang wajar 18–100 tahun) dengan imputasi median.
4. Menangani nilai `'unknown'` pada kolom kategorikal `job` dan `education` menggunakan imputasi modus.
5. Membatasi pencilan ekstrem (*outlier winsorization / clipping*) berbasis batas IQR pada kolom numerik.
6. Menerapkan *One-Hot Encoding* untuk mengonversi seluruh fitur kategorikal ke representasi biner.
7. Membagi data menjadi data latih (80%) dan data uji (20%) dengan *seed* terkontrol (`random_state=43`).
8. Melakukan standardisasi fitur manual (*Z-Score Scaling*) berbasis parameter data latih guna mencegah kebocoran data (*zero data leakage*).

### 4.1 Melihat Dimensi Data Sebelum Penghapusan

In [ ]:
print(f"Dimensi Data: {df.shape}")

### 4.2 Memeriksa Missing Value dan Duplikat Awal

In [ ]:
print("--- Missing Value pada Dataset ---")
print(df.isnull().sum())

In [ ]:
print(f"Jumlah Duplikat Data: {df.duplicated().sum()}")

### 4.3 Menghapus Kolom `Id` dan Menangani Baris Kosong

In [ ]:
df.drop(columns=['Id'], inplace=True)

In [ ]:
df.dropna(inplace=True)
df.reset_index(drop=True, inplace=True)

### 4.4 Verifikasi Status Data Setelah Penghapusan Kolom dan Missing Value

In [ ]:
print("Missing Value Setelah Dihapus:")
print(df.isnull().sum())

In [ ]:
print(f"Dimensi Data Setelah Penghapusan Kolom Id: {df.shape}")

### 4.5 Melihat Usia yang Anomali (Usia Normal = 18–100) $\rightarrow$ Mengubah Usia yang Anomali dengan Imputasi Median

In [ ]:
df['age'] = pd.to_numeric(df['age'], errors='coerce')
df['balance'] = pd.to_numeric(df['balance'], errors='coerce')

In [ ]:
kondisi = (df['age'] < 18) | (df['age'] > 100)
baris = df[kondisi]

print(f"Total baris anomali: {len(baris)}")
print(baris['age'])

In [ ]:
median_umur = df.loc[(df['age'] >= 18) & (df['age'] <= 100), 'age'].median()
df.loc[(df['age'] < 18) | (df['age'] > 100), 'age'] = median_umur

sisa = df[(df['age'] < 18) | (df['age'] > 100)]
print(f"Jumlah anomali umur setelah imputasi: {len(sisa)}")

### 4.6 Melihat Jumlah Unknown pada Kolom `job` dan `education` $\rightarrow$ Mengubah Unknown dengan Imputasi Modus

In [ ]:
for col in ['job', 'education']:
    modus = df[df[col] != 'unknown'][col].mode()[0]
    df[col] = df[col].replace('unknown', modus)

In [ ]:
print("Jumlah 'unknown' pada job:", (df['job'] == 'unknown').sum())
print("Jumlah 'unknown' pada education:", (df['education'] == 'unknown').sum())

### 4.7 Melihat Jumlah Outlier pada Kolom `age`, `balance`, `duration`, dan `campaign` $\rightarrow$ Membatasi Jumlah Outlier dengan Batas Minimum dan Maksimum IQR

In [ ]:
fitur_numerik = ['age', 'balance', 'duration', 'campaign']

for col in fitur_numerik:
    Q1 = df[col].quantile(0.25)
    Q3 = df[col].quantile(0.75)
    IQR = Q3 - Q1
    batas_bawah = Q1 - 1.5 * IQR
    batas_atas = Q3 + 1.5 * IQR
    
    outlier_awal = ((df[col] < batas_bawah) | (df[col] > batas_atas)).sum()
    df[col] = np.clip(df[col], batas_bawah, batas_atas)
    outlier_akhir = ((df[col] < batas_bawah) | (df[col] > batas_atas)).sum()
    
    print(f"{col:10} | Outlier Awal: {outlier_awal:4} -> Sisa Outlier: {outlier_akhir}")

### 4.8 Pemisahan Fitur ($X$) dan Target ($y$) serta One-Hot Encoding

In [ ]:
X = df.drop(columns=['y'])
y = df['y'].astype(str).str.strip().map({'no': 0, 'yes': 1})

X = pd.get_dummies(X, drop_first=True, dtype=int)

### 4.9 Pembagian Data Latih dan Data Uji (80% : 20%)

In [ ]:
np.random.seed(43)
indeks_acak = np.random.permutation(len(X))
batas_latih = int(0.8 * len(X))

X_train = X.iloc[indeks_acak[:batas_latih]].copy()
X_test  = X.iloc[indeks_acak[batas_latih:]].copy()
y_train = y.iloc[indeks_acak[:batas_latih]].copy()
y_test  = y.iloc[indeks_acak[batas_latih:]].copy()

print(f"Data Latih (X_train): {X_train.shape[0]} baris")
print(f"Data Uji   (X_test) : {X_test.shape[0]} baris")

### 4.10 Manual Standardization (Z-Score Scaling)

In [ ]:
rata_rata = X_train.mean()
standar_deviasi = X_train.std()

standar_deviasi[standar_deviasi == 0] = 1.0

X_train_scaled = (X_train - rata_rata) / standar_deviasi
X_test_scaled  = (X_test - rata_rata) / standar_deviasi

---

## 5. Modelling — K-Nearest Neighbors (KNN From Scratch)

Algoritma **K-Nearest Neighbors (KNN)** bekerja berdasarkan prinsip kemiripan spasial di mana data yang memiliki karakteristik mirip cenderung memiliki label kelas yang sama

### 5.1 Menentukan Nilai K Secara Langsung

In [ ]:
k = 5
print(f"Nilai K adalah: {k}")

### 5.2 Mengukur Jarak dengan Menggunakan Metrik Euclidean Distance

In [ ]:
def hitung_euclidean(nasabah_baru, X_latih):
    jarak = np.sqrt(np.sum((X_latih - nasabah_baru) ** 2, axis=1))
    return jarak

### 5.3 Simulasi Jarak Nasabah Uji ke-1 ke Seluruh Data Latih

In [ ]:
data_latih_np = X_train_scaled.values
nasabah_uji_1 = X_test_scaled.iloc[0].values

hasil_jarak = hitung_euclidean(nasabah_uji_1, data_latih_np)

print("Total jarak yang berhasil dihitung :", len(hasil_jarak))
print(f"{k} Jarak nasabah yang paling dekat :", np.sort(hasil_jarak)[:k])

### 5.4 Penentuan Tetangga Terdekat, Voting Mayoritas, dan Prediksi Nasabah Uji ke-1

In [ ]:
indeks = np.argsort(hasil_jarak)[:k]

target_latih = y_train.values
label = target_latih[indeks]

hasil = 1 if np.mean(label) >= 0.5 else 0

print(f"Indeks {k} nasabah terdekat di data latih     : {indeks}")
print(f"Label asli dari {k} tetangga tersebut         : {label}")
print(f"Hasil tebakan model untuk nasabah uji ke-1  : {hasil} ({'yes' if hasil == 1 else 'no'})")
print(f"Keputusan asli yang sebenarnya              : {y_test.iloc[0]} ({'yes' if y_test.iloc[0] == 1 else 'no'})")

### 5.5 Penyelesaian Pemodelan Penuh: Prediksi Seluruh Data Uji

In [ ]:
def knn_predict_fast(X_tr, y_tr, X_te, k=5, weights='uniform', chunk_size=1000):
    X_tr_mat = X_tr.values.astype(np.float32)
    X_te_mat = X_te.values.astype(np.float32)
    y_tr_mat = y_tr.values
    n_test = len(X_te_mat)
    
    preds = np.zeros(n_test, dtype=int)
    train_sq = np.sum(X_tr_mat**2, axis=1)
    
    for start in range(0, n_test, chunk_size):
        end = min(start + chunk_size, n_test)
        chunk = X_te_mat[start:end]
        test_sq = np.sum(chunk**2, axis=1, keepdims=True)
        dists_sq = np.maximum(0, test_sq - 2 * (chunk @ X_tr_mat.T) + train_sq)
        top_k_idx = np.argpartition(dists_sq, k, axis=1)[:, :k]
        
        if weights == 'uniform':
            top_k_labels = y_tr_mat[top_k_idx]
            preds[start:end] = (np.mean(top_k_labels, axis=1) >= 0.5).astype(int)
        else:
            row_idx = np.arange(len(chunk))[:, None]
            k_dists = np.sqrt(dists_sq[row_idx, top_k_idx])
            w = 1.0 / (k_dists + 1e-9)
            labels = y_tr_mat[top_k_idx]
            w_0 = np.sum(w * (labels == 0), axis=1)
            w_1 = np.sum(w * (labels == 1), axis=1)
            preds[start:end] = (w_1 >= w_0).astype(int)
            
    return preds

### 5.6 Eksekusi Prediksi Data Uji untuk Model Utama

In [ ]:
y_pred_k5 = knn_predict_fast(X_train_scaled, y_train, X_test_scaled, k=5, weights='uniform')
print("Prediksi seluruh data uji (9.043 nasabah) berhasil diselesaikan.")
print("Distribusi Kelas Prediksi Model :", pd.Series(y_pred_k5).value_counts().to_dict())

---

## 6. Evaluation

Pada permasalahan klasifikasi biner tak seimbang (*imbalanced dataset*), metrik akurasi semata dapat menimbulkan bias yang menyesatkan (**The Accuracy Paradox**). Oleh karena itu, evaluasi komprehensif wajib melibatkan metrik turunan *Confusion Matrix*:
* **True Positive (TP)**: Nasabah yang diprediksi berlangganan dan memang benar berlangganan.
* **True Negative (TN)**: Nasabah yang diprediksi tidak berlangganan dan memang benar tidak berlangganan.
* **False Positive (FP)**: Nasabah yang diprediksi berlangganan namun aslinya tidak (*Type I Error*).
* **False Negative (FN)**: Nasabah yang diprediksi tidak berlangganan padahal aslinya berlangganan (*Type II Error*).

Formulasi Metrik:
$$\text{Accuracy} = \frac{TP + TN}{TP + TN + FP + FN}$$
$$\text{Precision} = \frac{TP}{TP + FP}$$
$$\text{Recall} = \frac{TP}{TP + FN}$$
$$\text{F1-Score} = 2 \times \frac{\text{Precision} \times \text{Recall}}{\text{Precision} + \text{Recall}}$$

### 6.1 Perhitungan Metrik Evaluasi Model Utama

In [ ]:
def hitung_evaluasi_lengkap(y_true, y_pred):
    tp = int(np.sum((y_true == 1) & (y_pred == 1)))
    tn = int(np.sum((y_true == 0) & (y_pred == 0)))
    fp = int(np.sum((y_true == 0) & (y_pred == 1)))
    fn = int(np.sum((y_true == 1) & (y_pred == 0)))
    
    acc = (tp + tn) / len(y_true)
    prec = tp / (tp + fp) if (tp + fp) > 0 else 0.0
    rec = tp / (tp + fn) if (tp + fn) > 0 else 0.0
    f1 = 2 * (prec * rec) / (prec + rec) if (prec + rec) > 0 else 0.0
    
    return tn, fp, fn, tp, acc, prec, rec, f1

tn_5, fp_5, fn_5, tp_5, acc_5, prec_5, rec_5, f1_5 = hitung_evaluasi_lengkap(y_test, y_pred_k5)

eval_k5_df = pd.DataFrame([{
    'Model & Konfigurasi': 'KNN (K = 5, Uniform Voting)',
    'Accuracy': f"{acc_5 * 100:.2f}%",
    'Precision': f"{prec_5 * 100:.2f}%",
    'Recall': f"{rec_5 * 100:.2f}%",
    'F1-Score': f"{f1_5 * 100:.2f}%",
    'True Negative (TN)': tn_5,
    'False Positive (FP)': fp_5,
    'False Negative (FN)': fn_5,
    'True Positive (TP)': tp_5
}])
eval_k5_df

### 6.2 Eksperimen Pemilihan Nilai K Terbaik & Variasi Pembobotan

In [ ]:
daftar_eksperimen = [
    (1, 'uniform'),
    (3, 'uniform'),
    (5, 'uniform'),
    (7, 'uniform'),
    (9, 'uniform'),
    (3, 'distance'),
    (5, 'distance'),
    (7, 'distance')
]

hasil_eksperimen = []
for k_val, w_val in daftar_eksperimen:
    pred_k = knn_predict_fast(X_train_scaled, y_train, X_test_scaled, k=k_val, weights=w_val)
    tn_k, fp_k, fn_k, tp_k, acc_k, prec_k, rec_k, f1_k = hitung_evaluasi_lengkap(y_test, pred_k)
    hasil_eksperimen.append({
        'Nilai K': k_val,
        'Skema Voting': w_val.capitalize(),
        'Accuracy': f"{acc_k * 100:.2f}%",
        'Precision': f"{prec_k * 100:.2f}%",
        'Recall': f"{rec_k * 100:.2f}%",
        'F1-Score': f"{f1_k * 100:.2f}%",
        'TN': tn_k,
        'FP': fp_k,
        'FN': fn_k,
        'TP': tp_k
    })

tabel_komparasi = pd.DataFrame(hasil_eksperimen)
tabel_komparasi

### 6.3 Visualisasi Evaluasi Model KNN: Confusion Matrix & Kurva Tuning Nilai K

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(18, 6.5), dpi=115)

cm_k5 = np.array([[tn_5, fp_5], [fn_5, tp_5]])
sns.heatmap(cm_k5, annot=True, fmt='d', cmap='Blues', ax=axes[0], cbar=False,
            annot_kws={'size': 14, 'weight': 'bold'},
            xticklabels=['Pred: No (0)', 'Pred: Yes (1)'],
            yticklabels=['Aktual: No (0)', 'Aktual: Yes (1)'])
axes[0].set_title(f'Confusion Matrix KNN (K = 5, Uniform)\nAcc = {acc_5*100:.2f}% | Prec = {prec_5*100:.2f}%', fontsize=13, fontweight='bold', pad=12)
axes[0].set_xlabel('Prediksi Model', fontsize=11)
axes[0].set_ylabel('Aktual Kelas', fontsize=11)

k_list_plot = [1, 3, 5, 7, 9]
acc_vals = [float(row['Accuracy'].replace('%', '')) for row in hasil_eksperimen[:5]]
prec_vals = [float(row['Precision'].replace('%', '')) for row in hasil_eksperimen[:5]]
rec_vals = [float(row['Recall'].replace('%', '')) for row in hasil_eksperimen[:5]]
f1_vals = [float(row['F1-Score'].replace('%', '')) for row in hasil_eksperimen[:5]]

axes[1].plot(k_list_plot, acc_vals, marker='o', linewidth=2.5, color='tab:blue', label='Accuracy')
axes[1].plot(k_list_plot, prec_vals, marker='s', linewidth=2.5, color='tab:green', label='Precision')
axes[1].plot(k_list_plot, rec_vals, marker='^', linewidth=2.5, color='tab:red', label='Recall')
axes[1].plot(k_list_plot, f1_vals, marker='d', linewidth=2.5, color='tab:purple', label='F1-Score')

axes[1].scatter([5], [acc_vals[2]], color='tab:blue', s=130, zorder=5)
axes[1].scatter([5], [prec_vals[2]], color='tab:green', s=130, zorder=5)

axes[1].set_title('Kurva Tuning Parameter Nilai K terhadap Metrik Evaluasi', fontsize=13, fontweight='bold', pad=12)
axes[1].set_xlabel('Nilai Parameter K (Tetangga Terdekat)', fontsize=11)
axes[1].set_ylabel('Skor Metrik (%)', fontsize=11)
axes[1].set_xticks(k_list_plot)
axes[1].set_ylim(20, 100)
axes[1].legend(frameon=True, facecolor='white', framealpha=0.9)

plt.tight_layout()
plt.show()

---

## 7. Conclusion

### 7.1 Kesimpulan Analisis & Pemodelan
1. **Keberhasilan Pembangunan Pipeline Klasifikasi End-to-End**:
   Pengerjaan proyek berhasil menyelesaikan seluruh siklus data science mulai dari Business Understanding, Data Preparation, EDA komprehensif, pembersihan anomali data, implementasi KNN *from scratch* berbasis jarak Euclidean, hingga evaluasi menyeluruh pada 9.043 data uji.
2. **Pentingnya Kualitas Data & Feature Scaling pada KNN**:
   Koreksi usia anomali (imputasi median 39 tahun), *winsorization clipping* pada 4 fitur numerik, serta *Manual Standardization* terbukti sangat krusial dalam menyamakan skala fitur sehingga perhitungan jarak Euclidean tidak didistorsi oleh variabel berskala besar seperti `balance` dan `duration`.
3. **Kinerja Model Optimal pada $K = 5$**:
   Pengujian membuktikan bahwa model KNN dengan $K = 5$ (*Uniform Voting*) adalah model paling seimbang untuk operasional perbankan dengan **Akurasi 89.17%**, **Presisi 60.57%**, dan **F1-Score 40.85%**.